# Desafio 3 - ZettaLab

<img src="https://ufla.br/images/noticias/2020/04_abr/logo_zetta.png">

## Descrição do Problema

## Descrição da Solução

## Equipe

- Luciana Laibe Santos Silva, Comunicação e Marketing (9° Período, Graduação em Ciências Biológicas)
- Estevão Augusto da Fonseca Santos, Ciência e Governança de Dados (6° Período, Graduação em Ciência de Computação)
- Hugo Dias Pontello, Desenvolvimento de Software (5° Período, Graduação em Sistemas de Informação)
- Lorrana Verdi Flores, Desenvolvimento de Software (6° Período, Doutorado em Biotecnologia Vegetal)
- Bruna Oliveira Pereira, Geotecnologia (4° Período, Graduação em Engenharia Florestal)
- Geovanna Alexandre Possidonio, Gestão de Projetos (4° Período, Graduação em Administração)

## Importação de Bibliotecas

In [1]:
import pandas as pd             # Biblioteca para manipulação e análise de dados
import numpy as np              # Biblioteca para calculo de vetores e matrizes
import sys                      # Biblioteca para acessar variaveis e funções que interagem fortemente com o interpretador
import os                       # Biblioteca para interação com arquivos e diretórios a nivel sistema operacional
import basedosdados as bd       # Biblioteca para acessar o datalake público do site BasedosDados
import requests                 # Biblioteca para realizar requisiçoes HTTP
from pathlib import Path        # Biblioteca para a manipulação de caminhos do sistema a nivel orientado a objetos
import gzip                     # Biblioteca para compressão e decompressão de dados usando o formato gzip
import shutil                   # Biblioteca para manipulação de arquivos e diretorios

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import config_path          # Módulo que salva todos os caminhos de diretórios utilizados no projeto

from scripts import features        # Módulo de criação e manipulação de features
from scripts import modeling        # Módulo de modelagem de IA
from scripts import utils           # Módulo de utilitários genéricos
from scripts import pre_processing  # Módulo de pré-processamento e obtenção de dados.

from dotenv import load_dotenv      # Biblioteca para carregar variáveis de ambiente de arquivos .env
load_dotenv()
GOOGLE_CLOUD_ID_PROJECT = os.getenv("GOOGLE_CLOUD_ID_PROJECT") # Coloca o ID do projeto do Google Cloud numa constante

## Obtenção e Tratamento de Dados

#### [Banco de Dados de Queimadas - BaseDosDados](https://basedosdados.org/dataset/f06f3cdc-b539-409b-b311-1ff8878fb8d9?table=a3696dc2-4dd1-4f7e-9769-6aa16a1556b8)

Ocorrência do Fogo na Vegetação é o tema deste portal, http://queimadas.dgi.inpe.br/queimadas, desenvolvido no INPE, Instituto Nacional de Pesquisas Espaciais. Ele inclui o monitoramento operacional de focos de queimadas e de incêndios florestais detectados por satélites, e o cálculo e previsão do risco de fogo da vegetação.

Os dados para a América do Sul e a Central, África e Europa, são atualizados a cada três horas, todos os dias do ano. O acesso às informações é livre, e no navegador internet usado, deve estar desbloqueada a carga das janelas "pop up" para acesso a várias figuras, tabelas e gráficos.

Usuários que necessitam de focos com maior frequência temporal ou monitoramento em áreas específicas deverão consultar as "Perguntas Frequentes"

As informações neste portal estão divididas em blocos:

- "Situação Atual", é a "Sala de Situação" do Portal, e fornece para os últimos dois dias os resultados relevantes do monitoramento de focos de queima de vegetação feito pelo INPE em imagens satélites. Vários itens nos títulos das figuras e tabelas podem ser selecionados, alterando as apresentações gráficas conforme as preferências espaciais e temporais do usuário. Passando o mouse sobre os títulos e figuras serão indicadas as opções ativas.
- "SIG Focos-Geral", permite visualizar os focos em um Sistema de Informação Geográfica, com opções de períodos, regiões de interesse, satélites, planos de informação (p.ex. desmatamento, hidrografia, estradas), etc., além da exportação dos dados em formatos txt, html, shp e kmz.
- "SIG Focos-Áreas Protegidas", Semelhante ao item anterior, mas dedicado à ocorrência do fogo em Áreas de Preservação, como Parques, Florestas, Reservas Biológicas municipais, estaduais e nacionais, e Terras Indígenas.
- "Situação nas Áreas Protegidas", que contém o último relatório de focos detectados nas áreas de preservação, incluindo links que os mostram já inseridos no SIG do monitoramento.
- "Relatório Atual", contém o último resumo do monitoramento de queimadas em formato pdf, que poderá ser salvo pelo usuário para análise detalhada dos dados. A opção "Receber por E-Mail" leva ao cadastro do usuário, onde se define individualmente o conteúdo dos relatórios e mensagens de alerta que serão recebidos automaticamente por E-Mail.
- "Risco de Fogo", que apresenta o Risco de Fogo da vegetação estimado no presente, e suas previsões futuras (diárias, semanais e mensais), bem como o "fogograma" para qualquer local selecionado com o mouse nos mapas.
- "Meu Cadastro", onde o usuário define e configura individualmente os produtos que deseja receber em seu E-Mail, como alertas de ocorrência de focos em áreas de preservação, boletins pdf diários com tabelas, gráficos e mapas de estados e países de interesse, e mensagens operacionais.
- "Outros Produtos", com links a produtos adicionais gerados por este sistema de Queimadas e Incêndios, destacando-se: mapas mensais de focos em formato .gif com animações; mapas Meteorológicos (precipitação, temperatura, umidade do ar, ventos); detecções dos vários satélites utilizados; estimativas de concentrações e trajetórias dos poluentes emitidos; localização dos satélites utilizados, uma coletânea de dados anteriores e área de download, e; acesso a uma antiga versão da página do monitoramento de focos.
- "Links Relacionados", com indicações de acesso ao Boletim do Ibama de Monitoramento de Queimadas e Incêndios, ao texto publicado mensalmente no Boletim Climanálise, a uma coleção de mais de 500 páginas com matérias, fotos, vídeos e sites diversos sobre queimadas e incêndios florestais, à composição da equipe deste trabalho, e aos agradecimentos.
- "Notícias e Destaques", com informações, notícias e links de destaque na ocorrência do fogo na vegetação, e não restritas apenas à detecção de focos com satélites.
- "Aviso", com informações operacionais indicando novidades, falhas e detalhes relevantes na geração dos produtos do monitoramento do fogo na vegetação.

**Organização**

Instituto Nacional de Pesquisas Espaciais (INPE)

**Cobertura Temporal**

2000-2025

In [4]:
query = """
SELECT
    dados.ano as ano,
    dados.mes as mes,
    dados.data_hora as data_hora,
    dados.bioma as bioma,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.latitude as latitude,
    dados.longitude as longitude,
    dados.satelite as satelite,
    dados.dias_sem_chuva as dias_sem_chuva,
    dados.precipitacao as precipitacao,
    dados.risco_fogo as risco_fogo,
    dados.potencia_radiativa_fogo as potencia_radiativa_fogo
FROM `basedosdados.br_inpe_queimadas.microdados` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano = 2024
    """

caminho_completo = os.path.join(config_path.RAW_DATA_DIRECTORY_PATH, "banco_dados_queimadas_raw.csv")
if not os.path.exists(caminho_completo):
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    df.to_csv(caminho_completo, index=False)
    
df = pd.read_csv(caminho_completo)
df.head()

Downloading: 100%|██████████|


,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,8,2024-08-31 12:47:27,Mata Atlântica,SC,Santa Catarina,4205803,Garuva,-26.04640,-48.95320,GOES-16,-999.0,0.0,0.5,115.5
1,2024,8,2024-08-31 15:37:00,Mata Atlântica,ES,Espírito Santo,3201159,Brejetuba,-20.23871,-41.34359,NOAA-21,96.0,0.0,1.0,34.6
2,2024,8,2024-08-31 16:05:00,Caatinga,RN,Rio Grande do Norte,2407104,Macaíba,-6.02403,-35.50732,NPP-375,26.0,0.0,1.0,19.8
3,2024,8,2024-08-31 16:05:00,Caatinga,PE,Pernambuco,2612554,Santa Filomena,-8.33915,-40.53336,NPP-375,51.0,0.0,1.0,15.5
4,2024,8,2024-08-31 16:26:00,Mata Atlântica,ES,Espírito Santo,3201902,Domingos Martins,-20.31597,-41.08064,NOAA-20,21.0,0.0,1.0,14.2


#### [Programa de Queimadas - INPE](https://terrabrasilis.dpi.inpe.br/queimadas/portal/)

#### [Programa de Queimadas - INPE (BDQueimadas)](https://terrabrasilis.dpi.inpe.br/queimadas/bdqueimadas/#exportar-dados)

In [ ]:
url = "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=7b6873c5-ded3-50f2-a67a-07b0c1716dbb&token=476fc698-b359-5459-b529-196ec5db488a&id=ce43b6b8-bdf8-50fe-9ea0-6a46f1f922bc"


